In [6]:
import os
import cv2 as cv
import numpy as np
import pandas as pd 
import tensorflow as tf
import matplotlib.pyplot as plt

In [7]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

print("Path to dataset files:", path)

Path to dataset files: /Users/ayushprasad/.cache/kagglehub/datasets/masoudnickparvar/brain-tumor-mri-dataset/versions/2


In [10]:
x = []
y = []

train_dir = os.path.join(path, "Training")

for label in os.listdir(train_dir):

    label_path = os.path.join(train_dir, label)

    for file in os.listdir(label_path):

        file_path = os.path.join(label_path, file)

        img = cv.imread(file_path)        # BEST way
        img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        img = cv.resize(img, (128,128))

        x.append(img)
        y.append(label)

In [11]:
X_train=np.array(x)
y_train=np.array(y)

In [12]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y_train=le.fit_transform(y_train)

In [13]:
model=tf.keras.models.Sequential()

# CNN
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False
model.add(base_model)
model.add(tf.keras.layers.GlobalAveragePooling2D())
# ANN

model.add(tf.keras.layers.Dense(16,activation='relu'))
model.add(tf.keras.layers.Dense(16,activation='relu'))
model.add(tf.keras.layers.Dense(4,activation='softmax'))

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [14]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │        20,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,278,820 (8.69 MB)

 Trainable params: 20,836 (81.39 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [15]:
@tf.function
def train_step(X, y):
    with tf.GradientTape() as tape:
        pred = model(X, training=True)
        loss = loss_fn(y, pred)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

In [16]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

batch_size = 128
dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
dataset = dataset.shuffle(X_train.shape[0]).batch(batch_size).prefetch(tf.data.AUTOTUNE) 

epochs = 10
loss_val=[]
for i in range(epochs):
    for X_batch, y_batch in dataset:
        loss = train_step(X_batch, y_batch)
    loss_val.append(loss)
    print("Epoch:", i+1, "Loss:", loss.numpy())

2026-03-18 11:19:56.507345: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch: 1 Loss: 0.653614


2026-03-18 11:20:10.861978: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch: 2 Loss: 0.5025811
Epoch: 3 Loss: 0.46233106


2026-03-18 11:20:34.860139: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch: 4 Loss: 0.3367084
Epoch: 5 Loss: 0.4078952
Epoch: 6 Loss: 0.26375684
Epoch: 7 Loss: 0.29449186


2026-03-18 11:21:26.624666: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch: 8 Loss: 0.30532077
Epoch: 9 Loss: 0.35095394
Epoch: 10 Loss: 0.31108746


In [17]:
accuracy=np.mean(np.argmax(model.predict(X_train),axis=1)==y_train)
print("Training Accuracy:", accuracy*100)

175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 73ms/step
Training Accuracy: 91.64285714285715


In [19]:
x = []
y = []

test_dir = os.path.join(path, "Testing")

for label in os.listdir(test_dir):

    label_path = os.path.join(test_dir, label)

    for file in os.listdir(label_path):

        file_path = os.path.join(label_path, file)

        img = cv.imread(file_path)
        img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        img = cv.resize(img, (128,128))

        x.append(img)
        y.append(label)

In [20]:
X_test=np.array(x)
y_test=np.array(y)

In [21]:
y_test=le.transform(y_test)

In [23]:
accuracy=np.mean(np.argmax(model.predict(X_test),axis=1)==y_test)
print("Testing Accuracy:", accuracy*100)

50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 70ms/step
Testing Accuracy: 80.875
